In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit

spark = SparkSession.builder \
    .appName("Tugas6_ETL") \
    .getOrCreate()

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [4]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


A. EXTRACT (bobot 15%)

Baca ketiga sumber data (tugas6_transaksi.csv, tugas6_produk.json, tugas6_ulasan.csv) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan printSchema() masing-masing.

Membaca Data Transaksi

In [5]:
df_trx = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("tugas6_transaksi.csv")

print("Jumlah data transaksi:", df_trx.count())
df_trx.printSchema()
df_trx.show(5)

Jumlah data transaksi: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

+--------+----------+------------+-------------------+
|order_id|product_id|unit_terjual|            tanggal|
+--------+----------+------------+-------------------+
|     TX0|        21|           2|2026-10-10 00:00:00|
|     TX1|         8|           6|2026-10-25 00:00:00|
|     TX2|        25|           4|2026-10-13 00:00:00|
|     TX3|         3|           4|2026-10-18 00:00:00|
|     TX4|        19|           6|2026-10-07 00:00:00|
+--------+----------+------------+-------------------+
only showing top 5 rows



Membaca Data Produk

In [6]:
df_produk = spark.read \
    .option("inferSchema", True) \
    .json("tugas6_produk.json")

print("Jumlah data produk:", df_produk.count())
df_produk.printSchema()
df_produk.show(5)

Jumlah data produk: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

+------+------------+-----------+----------+
| harga|    kategori|nama_produk|product_id|
+------+------------+-----------+----------+
| 75000|Rumah Tangga|   Produk-1|         1|
|250000|  Elektronik|   Produk-2|         2|
| 75000|     Fashion|   Produk-3|         3|
|150000|     Makanan|   Produk-4|         4|
| 75000|   Kesehatan|   Produk-5|         5|
+------+------------+-----------+----------+
only showing top 5 rows



Membaca Data Ulasan

In [7]:
df_review = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("tugas6_ulasan.csv")

print("Jumlah data ulasan:", df_review.count())
df_review.printSchema()
df_review.show(5)

Jumlah data ulasan: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)

+--------+------+
|order_id|rating|
+--------+------+
|  TX3595|     1|
|  TX2740|     5|
|  TX4997|     1|
|  TX2758|     2|
|  TX1179|     4|
+--------+------+
only showing top 5 rows



B. TRANSFORM — Penggabungan (bobot 25%)

Gabungkan ketiga DataFrame menjadi satu (order_id sebagai kunci ke ulasan, product_id sebagai kunci ke produk). Gunakan salah satu join yang tepat dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan salah satu join yang tepat dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom total_pendapatan (unit_terjual x harga).

Join Transaksi dengan Ulasan

In [8]:
df_gabung_review = df_trx.join(
    df_review,
    on="order_id",
    how="left"
)

df_gabung_review.show(5)

+--------+----------+------------+-------------------+------+
|order_id|product_id|unit_terjual|            tanggal|rating|
+--------+----------+------------+-------------------+------+
|     TX0|        21|           2|2026-10-10 00:00:00|     1|
|     TX1|         8|           6|2026-10-25 00:00:00|  NULL|
|     TX2|        25|           4|2026-10-13 00:00:00|     2|
|     TX3|         3|           4|2026-10-18 00:00:00|     5|
|     TX4|        19|           6|2026-10-07 00:00:00|  NULL|
+--------+----------+------------+-------------------+------+
only showing top 5 rows



Join dengan Data Produk

In [9]:
df_gabung = df_gabung_review.join(
    df_produk,
    on="product_id",
    how="inner"
)

df_gabung.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|
+----------+--------+------------+-------------------+------+------+------------+-----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|
+----------+--------+------------+-------------------+------+------+------------+-----------+
only showing top 5 rows



Menambahkan Total Pendapatan

In [10]:
df_gabung = df_gabung.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

df_gabung.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
o

C. TRANSFORM — Penanganan Data Kosong & Pengayaan (bobot 20%)

    Transaksi tanpa ulasan akan memiliki rating bernilai kosong (null) setelah salah satu join yang tepat — isi nilai kosong tersebut dengan angka 0 menggunakan salah satu function, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
    Tambahkan kolom ada_ulasan bernilai True/False (tidak boleh diisi manual satu satu), **sebelum** langkah na.fill()` di atas).


Membuat kolom ada_ulasan

In [13]:
df_gabung = df_gabung.withColumn(
    "ada_ulasan",
    col("rating").isNotNull()
)

df_gabung.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|     false|
+----------+--------+---

Mengisi Rating yang Kosong

In [17]:
df_gabung = df_gabung.na.fill(
    {"rating": 0}
)

df_gabung.printSchema()
df_gabung.show(10, truncate=False)

root
 |-- product_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- rating: integer (nullable = false)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- total_pendapatan: long (nullable = true)
 |-- ada_ulasan: boolean (nullable = false)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|tanggal            |rating|harga |kategori    |nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|21        |TX0     |2           |2026-10-10 00:00:00|1     |25000 |Elektronik  |Produk-21  |50000           |true      |
|8         |TX1     |6           |2026-10-25 00:00:00|0     |250000|Fashion     |Produk-

D. LOAD (bobot 25%)

Simpan hasil akhir ke HDFS dalam format Parquet, dipartisi berdasarkan kategori, ke path /user/[username]/tugas6/hasil_etl. Verifikasi dengan hdfs dfs -ls -R, lalu baca kembali dan tampilkan count()-nya sebagai bukti data tersimpan utuh.

In [18]:
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6

df_gabung.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"
)

print("Pipeline ETL selesai — hasil tersimpan di HDFS.")

!hdfs dfs -ls /user/mahasiswa/tugas6/hasil_etl

[Stage 48:>                                                         (0 + 1) / 1]

Pipeline ETL selesai — hasil tersimpan di HDFS.


Found 6 items
-rw-r--r--   3 manggala supergroup          0 2026-09-30 23:24 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - manggala supergroup          0 2026-09-30 23:24 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
drwxr-xr-x   - manggala supergroup          0 2026-09-30 23:24 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
drwxr-xr-x   - manggala supergroup          0 2026-09-30 23:24 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
drwxr-xr-x   - manggala supergroup          0 2026-09-30 23:24 /user/mahasiswa/tugas6/hasil_etl/kategori=Makanan
drwxr-xr-x   - manggala supergroup          0 2026-09-30 23:24 /user/mahasiswa/tugas6/hasil_etl/kategori=Rumah Tangga


In [19]:
df_hasil = spark.read.parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"
)

print("Jumlah data setelah dibaca kembali:", df_hasil.count())

Jumlah data setelah dibaca kembali: 5000


E. Insight Akhir (bobot 15%)

Dari data hasil ETL, tampilkan (menggunakan DataFrame API atau Spark SQL, bebas memilih): kategori produk mana yang memiliki persentase transaksi dengan ulasan (ada_ulasan = True) paling rendah? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

In [21]:
from pyspark.sql.functions import count, sum, round

df_insight = df_hasil.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    sum(
        when(col("ada_ulasan") == True, 1).otherwise(0)
    ).alias("transaksi_dengan_ulasan")
)

df_insight = df_insight.withColumn(
    "persentase_ulasan",
    round(
        col("transaksi_dengan_ulasan") /
        col("total_transaksi") * 100,
        2
    )
)

df_insight.orderBy("persentase_ulasan").show()

+------------+---------------+-----------------------+-----------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|persentase_ulasan|
+------------+---------------+-----------------------+-----------------+
|     Makanan|            536|                    369|            68.84|
|   Kesehatan|            961|                    662|            68.89|
|     Fashion|           1035|                    726|            70.14|
|Rumah Tangga|            815|                    575|            70.55|
|  Elektronik|           1653|                   1168|            70.66|
+------------+---------------+-----------------------+-----------------+



**Interpretasi Bisnis**

Berdasarkan hasil analisis, kategori Makanan memiliki persentase transaksi dengan ulasan paling rendah. Kondisi ini penting diketahui oleh tim marketing karena rendahnya jumlah ulasan dapat menunjukkan bahwa interaksi pelanggan setelah pembelian masih perlu ditingkatkan. Tim marketing dapat menggunakan informasi ini sebagai bahan evaluasi untuk mendorong pelanggan memberikan ulasan setelah melakukan transaksi.
